# Numerical Missing Value Imputation

## Objective

In this notebook, we will:

- Preserve the original Heart Failure dataset
- Introduce numerical missing values in a practice copy
- Apply Mean Imputation
- Apply Median Imputation
- Apply Mode Imputation
- Apply Conditional Mean Imputation
- Apply KNN Imputation
- Apply Regression Imputation
- Compare imputed values
- Verify that no missing values remain in the selected features



In [4]:
# Import required libraries
import numpy as np
import pandas as pd

from sklearn.impute import KNNImputer
from sklearn.linear_model import LinearRegression

In [5]:
# Load the original Heart Failure dataset
df = pd.read_csv(
    "../heart_failure_clinical_records_dataset-selected-columns.csv"
)

df.head()

,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex
0,75.0,0,582,0,20,1,265000.00,1.9,130,1
1,55.0,0,7861,0,38,0,263358.03,1.1,136,1
2,65.0,0,146,0,20,0,162000.00,1.3,129,1
3,50.0,1,111,0,20,0,210000.00,1.9,137,1
4,65.0,1,160,1,20,0,327000.00,2.7,116,0


In [6]:
# Verify the original dataset
print("Dataset Shape:", df.shape)

print(
    "Original Missing Values:",
    df.isnull().sum().sum()
)

Dataset Shape: (299, 10)
Original Missing Values: 0


## Create Numerical Missing Values

The original dataset contains no missing values.

A separate copy will be created, and missing values will be introduced only for practical learning.

In [7]:
# Create an independent practice copy
df_missing = df.copy()

# Missing values for Mean Imputation
df_missing.loc[
    [5, 10, 15, 20, 25],
    "age"
] = np.nan

# Missing values for Median Imputation
df_missing.loc[
    [30, 35, 40, 45, 50],
    "serum_creatinine"
] = np.nan

# Missing values for Mode Imputation
df_missing.loc[
    [55, 60, 65, 70],
    "ejection_fraction"
] = np.nan

# Display missing counts
df_missing[
    [
        "age",
        "serum_creatinine",
        "ejection_fraction"
    ]
].isnull().sum()

age                  5
serum_creatinine     5
ejection_fraction    4
dtype: int64

In [8]:
# Store indexes for later comparison

age_missing_indexes = (
    df_missing[
        df_missing["age"].isnull()
    ].index
)

creatinine_missing_indexes = (
    df_missing[
        df_missing["serum_creatinine"].isnull()
    ].index
)

ejection_missing_indexes = (
    df_missing[
        df_missing["ejection_fraction"].isnull()
    ].index
)

print("Age Missing Indexes:", age_missing_indexes.tolist())

print(
    "Serum Creatinine Missing Indexes:",
    creatinine_missing_indexes.tolist()
)

print(
    "Ejection Fraction Missing Indexes:",
    ejection_missing_indexes.tolist()
)

Age Missing Indexes: [5, 10, 15, 20, 25]
Serum Creatinine Missing Indexes: [30, 35, 40, 45, 50]
Ejection Fraction Missing Indexes: [55, 60, 65, 70]


# Mean Imputation

Mean Imputation replaces missing values with the average of the available values.

It is most suitable for approximately symmetric numerical data without strong outliers.

In [9]:
# Calculate the mean of Age
age_mean = df_missing["age"].mean()

print(f"Age Mean: {age_mean:.2f}")

Age Mean: 60.54


In [10]:
# Create a separate copy for Mean Imputation
df_mean = df_missing.copy()

# Fill missing Age values with the mean
df_mean["age"] = df_mean["age"].fillna(
    age_mean
)

# Display imputed rows
df_mean.loc[
    age_missing_indexes,
    ["age"]
]

,age
5,60.53515
10,60.53515
15,60.53515
20,60.53515
25,60.53515


In [11]:
print(
    "Remaining Missing Age Values:",
    df_mean["age"].isnull().sum()
)

Remaining Missing Age Values: 0


# Median Imputation

Median Imputation replaces missing values with the middle value of the feature.

It is suitable for skewed data and features containing outliers.

In [12]:
# Calculate median Serum Creatinine
creatinine_median = (
    df_missing["serum_creatinine"]
    .median()
)

print(
    "Serum Creatinine Median:",
    creatinine_median
)

Serum Creatinine Median: 1.1


In [13]:
# Create a separate copy for Median Imputation
df_median = df_missing.copy()

df_median["serum_creatinine"] = (
    df_median["serum_creatinine"]
    .fillna(creatinine_median)
)

df_median.loc[
    creatinine_missing_indexes,
    ["serum_creatinine"]
]

,serum_creatinine
30,1.1
35,1.1
40,1.1
45,1.1
50,1.1


In [14]:
print(
    "Remaining Missing Serum Creatinine Values:",
    df_median["serum_creatinine"]
    .isnull()
    .sum()
)

Remaining Missing Serum Creatinine Values: 0


# Mode Imputation

Mode Imputation replaces missing values with the most frequently occurring value.

It is useful for discrete numerical variables.

In [15]:
# Calculate the most frequent Ejection Fraction value
ejection_mode = (
    df_missing["ejection_fraction"]
    .mode()[0]
)

print(
    "Ejection Fraction Mode:",
    ejection_mode
)

Ejection Fraction Mode: 35.0


In [16]:
# Create a separate copy for Mode Imputation
df_mode = df_missing.copy()

df_mode["ejection_fraction"] = (
    df_mode["ejection_fraction"]
    .fillna(ejection_mode)
)

df_mode.loc[
    ejection_missing_indexes,
    ["ejection_fraction"]
]

,ejection_fraction
55,35.0
60,35.0
65,35.0
70,35.0


In [17]:
print(
    "Remaining Missing Ejection Fraction Values:",
    df_mode["ejection_fraction"]
    .isnull()
    .sum()
)

Remaining Missing Ejection Fraction Values: 0


# Conditional Mean Imputation

Conditional Mean Imputation fills missing values using the mean of a meaningful group.

For practice, missing Age values will be filled using the mean Age within each `diabetes` group.

In [18]:
# Create a separate copy
df_conditional = df_missing.copy()

# Fill missing Age values using the
# mean Age of each diabetes group
df_conditional["age"] = (
    df_conditional
    .groupby("diabetes")["age"]
    .transform(
        lambda group: group.fillna(
            group.mean()
        )
    )
)

df_conditional.loc[
    age_missing_indexes,
    ["diabetes", "age"]
]

,diabetes,age
5,0,61.470588
10,0,61.470588
15,0,61.470588
20,0,61.470588
25,1,59.252694


In [19]:
# Display Age mean for each diabetes group
group_age_means = (
    df_missing
    .groupby("diabetes")["age"]
    .mean()
)

group_age_means

diabetes
0    61.470588
1    59.252694
Name: age, dtype: float64

In [20]:
print(
    "Remaining Missing Age Values:",
    df_conditional["age"]
    .isnull()
    .sum()
)

Remaining Missing Age Values: 0


# KNN Imputation

KNN Imputation estimates missing values using similar observations.

For demonstration, selected continuous numerical features will be used.

> KNN is distance-based and can be affected by different feature scales. Scaling will be covered in its dedicated topic.

In [21]:
# Select continuous numerical features
knn_features = [
    "age",
    "creatinine_phosphokinase",
    "ejection_fraction",
    "platelets",
    "serum_creatinine",
    "serum_sodium"
]

df_knn = df_missing.copy()

df_knn[knn_features].isnull().sum()

age                         5
creatinine_phosphokinase    0
ejection_fraction           4
platelets                   0
serum_creatinine            5
serum_sodium                0
dtype: int64

In [22]:
# Create KNN imputer
knn_imputer = KNNImputer(
    n_neighbors=5
)

# Fit and transform selected features
knn_imputed_array = knn_imputer.fit_transform(
    df_knn[knn_features]
)

# Convert result back into a DataFrame
df_knn_imputed = pd.DataFrame(
    knn_imputed_array,
    columns=knn_features,
    index=df_knn.index
)

df_knn_imputed.head()

,age,creatinine_phosphokinase,ejection_fraction,platelets,serum_creatinine,serum_sodium
0,75.0,582.0,20.0,265000.00,1.9,130.0
1,55.0,7861.0,38.0,263358.03,1.1,136.0
2,65.0,146.0,20.0,162000.00,1.3,129.0
3,50.0,111.0,20.0,210000.00,1.9,137.0
4,65.0,160.0,20.0,327000.00,2.7,116.0


In [23]:
# Replace numerical columns with imputed values
df_knn[knn_features] = df_knn_imputed

print(
    "Remaining Missing Values in KNN Features:",
    df_knn[knn_features]
    .isnull()
    .sum()
    .sum()
)

Remaining Missing Values in KNN Features: 0


In [24]:
# Display selected rows that originally contained missing values
all_missing_indexes = sorted(
    set(age_missing_indexes)
    | set(creatinine_missing_indexes)
    | set(ejection_missing_indexes)
)

df_knn.loc[
    all_missing_indexes,
    [
        "age",
        "ejection_fraction",
        "serum_creatinine"
    ]
]

,age,ejection_fraction,serum_creatinine
5,62.7334,40.0,2.100
10,53.8000,38.0,4.000
15,64.4000,50.0,1.300
20,58.4000,25.0,1.300
25,64.0000,38.0,1.900
30,94.0000,38.0,1.570
35,69.0000,35.0,1.430
40,70.0000,20.0,1.634
45,50.0000,38.0,1.120
50,68.0000,25.0,1.056


# Regression Imputation

Regression Imputation trains a model using complete rows and predicts the missing values.

In this example:

- Target feature: `serum_creatinine`
- Predictor features:
  - `age`
  - `ejection_fraction`
  - `platelets`
  - `serum_sodium`

In [25]:
# Create an independent copy
df_regression = df_missing.copy()

target_column = "serum_creatinine"

predictor_columns = [
    "age",
    "ejection_fraction",
    "platelets",
    "serum_sodium"
]

# Use only rows where predictors are complete
predictors_complete = (
    df_regression[predictor_columns]
    .notnull()
    .all(axis=1)
)

# Complete target rows for training
training_mask = (
    df_regression[target_column].notnull()
    & predictors_complete
)

# Missing target rows for prediction
prediction_mask = (
    df_regression[target_column].isnull()
    & predictors_complete
)

print(
    "Training Rows:",
    training_mask.sum()
)

print(
    "Rows to Predict:",
    prediction_mask.sum()
)

Training Rows: 285
Rows to Predict: 5


In [26]:
# Prepare training data
X_train = df_regression.loc[
    training_mask,
    predictor_columns
]

y_train = df_regression.loc[
    training_mask,
    target_column
]

# Create and train model
regression_model = LinearRegression()

regression_model.fit(
    X_train,
    y_train
)

print("Regression Model Trained")

Regression Model Trained


In [27]:
# Prepare rows with missing target values
X_missing = df_regression.loc[
    prediction_mask,
    predictor_columns
]

# Predict missing Serum Creatinine values
predicted_values = regression_model.predict(
    X_missing
)

predicted_values

array([1.86173321, 1.57519525, 1.55000725, 1.29563165, 1.41149048])

In [28]:
# Fill missing target values with predictions
df_regression.loc[
    prediction_mask,
    target_column
] = predicted_values

# Display imputed rows
df_regression.loc[
    prediction_mask,
    predictor_columns + [target_column]
]

,age,ejection_fraction,platelets,serum_sodium,serum_creatinine
30,94.0,38.0,263358.03,134,1.861733
35,69.0,35.0,228000.00,134,1.575195
40,70.0,20.0,263358.03,134,1.550007
45,50.0,38.0,310000.00,135,1.295632
50,68.0,25.0,166000.00,138,1.411490


In [29]:
print(
    "Remaining Missing Serum Creatinine Values:",
    df_regression[target_column]
    .isnull()
    .sum()
)

Remaining Missing Serum Creatinine Values: 0


# Compare Imputed Values

The following comparison shows how different methods can generate different replacement values for the same missing records.

In [30]:
# Compare Mean, Conditional Mean and KNN for missing Age
age_comparison = pd.DataFrame({
    "Original Missing": (
        df_missing.loc[
            age_missing_indexes,
            "age"
        ]
    ),
    "Mean Imputation": (
        df_mean.loc[
            age_missing_indexes,
            "age"
        ]
    ),
    "Conditional Mean": (
        df_conditional.loc[
            age_missing_indexes,
            "age"
        ]
    ),
    "KNN Imputation": (
        df_knn.loc[
            age_missing_indexes,
            "age"
        ]
    )
})

age_comparison

,Original Missing,Mean Imputation,Conditional Mean,KNN Imputation
5,NaN,60.53515,61.470588,62.7334
10,NaN,60.53515,61.470588,53.8000
15,NaN,60.53515,61.470588,64.4000
20,NaN,60.53515,61.470588,58.4000
25,NaN,60.53515,59.252694,64.0000


In [31]:
# Compare Median, KNN and Regression methods
creatinine_comparison = pd.DataFrame({
    "Original Missing": (
        df_missing.loc[
            creatinine_missing_indexes,
            "serum_creatinine"
        ]
    ),
    "Median Imputation": (
        df_median.loc[
            creatinine_missing_indexes,
            "serum_creatinine"
        ]
    ),
    "KNN Imputation": (
        df_knn.loc[
            creatinine_missing_indexes,
            "serum_creatinine"
        ]
    ),
    "Regression Imputation": (
        df_regression.loc[
            creatinine_missing_indexes,
            "serum_creatinine"
        ]
    )
})

creatinine_comparison

,Original Missing,Median Imputation,KNN Imputation,Regression Imputation
30,NaN,1.1,1.570,1.861733
35,NaN,1.1,1.430,1.575195
40,NaN,1.1,1.634,1.550007
45,NaN,1.1,1.120,1.295632
50,NaN,1.1,1.056,1.411490


In [32]:
# Verify the selected imputed copies

verification = pd.DataFrame({
    "Method": [
        "Mean",
        "Median",
        "Mode",
        "Conditional Mean",
        "KNN",
        "Regression"
    ],
    "Relevant Missing Values": [
        df_mean["age"].isnull().sum(),
        df_median["serum_creatinine"].isnull().sum(),
        df_mode["ejection_fraction"].isnull().sum(),
        df_conditional["age"].isnull().sum(),
        df_knn[knn_features].isnull().sum().sum(),
        df_regression["serum_creatinine"].isnull().sum()
    ]
})

verification

,Method,Relevant Missing Values
0,Mean,0
1,Median,0
2,Mode,0
3,Conditional Mean,0
4,KNN,0
5,Regression,0


In [33]:
print("Original Dataset Shape:", df.shape)

print(
    "Original Dataset Missing Values:",
    df.isnull().sum().sum()
)

Original Dataset Shape: (299, 10)
Original Dataset Missing Values: 0


# Summary

In this notebook, we:

- Preserved the original Heart Failure dataset
- Introduced numerical missing values in a practice copy
- Applied Mean Imputation
- Applied Median Imputation
- Applied Mode Imputation
- Applied Conditional Mean Imputation
- Applied KNN Imputation
- Applied Regression Imputation
- Compared replacement values generated by different techniques
- Verified that the original dataset remained unchanged

## Key Learnings

- Mean is suitable for approximately symmetric data.
- Median is more robust for skewed data and outliers.
- Mode is suitable for discrete numerical features.
- Conditional Mean preserves group-level differences.
- KNN uses similar observations.
- Regression predicts missing values using other features.
- Different methods produce different estimates.
- Imputation should be selected through analysis and validation.

## Next Topic

Categorical Missing Value Imputation